# 🚀 Out-of-Fold Model Training Pipeline

## Objective
This notebook implements a **production-grade training pipeline** using:

- Time-based cross-validation
- Out-of-Fold (OOF) predictions
- Multiple model families (LightGBM, XGBoost, CatBoost)
- Proper probability-based evaluation (F1 optimization ready)

## Why this matters
OOF predictions are critical for:
- Stacking
- Weighted ensembling
- Threshold optimization
- Reliable performance estimation

This notebook forms the backbone of the final ensemble system.

**1. IMPORTS & SETUP**

In [1]:
# =========================
# SETUP
# =========================

import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))

# Project modules
from src.modeling import run_time_cv_oof
from src.config import *

# Models
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Utils
from sklearn.metrics import f1_score

**2. LOAD PROCESSED DATA**

In [3]:
# =========================
# LOAD DATA
# =========================

train = pd.read_parquet("D:/AI4EAC- Loan_default_prediction/data/processed/train_merged.parquet")
test = pd.read_parquet("D:/AI4EAC- Loan_default_prediction/data/processed/test_merged.parquet")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (68654, 38)
Test shape: (18594, 29)


**3. FINAL FEATURE SET**

In [10]:
# =========================
# FEATURE SELECTION
# =========================

DROP_COLS = [
    "ID", "target", "customer_id",
    "tbl_loan_id", "lender_id",
    "disbursement_date", "due_date",
    "country_name", "country",

    # 🚨 REMOVE TRAIN-ONLY FEATURES
    "loan_count",
    "past_defaults",
    "total_loans",
    "safe_default_rate"
]

FEATURES = [col for col in train.columns if col not in DROP_COLS]

X = train[FEATURES].copy()
y = train["target"].copy()

# SAFE ALIGNMENT
X_test = test.reindex(columns=FEATURES, fill_value=0)

In [6]:
X.shape, X_test.shape

((68654, 27), (18594, 27))

In [12]:
# Convert category → numeric codes
for col in X.select_dtypes(include=['category']).columns:
    X[col] = X[col].cat.codes

for col in X_test.select_dtypes(include=['category']).columns:
    X_test[col] = X_test[col].cat.codes

#enforce numeric safety
X = X.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')

In [13]:
print(X.dtypes.value_counts())

float64    18
int8        3
int64       3
int32       2
Name: count, dtype: int64


**4. MISSING VALUE FINAL FIX**

In [ ]:
# =========================
# FINAL MISSING HANDLING
# =========================

def fill_missing(df):
    df = df.copy()
    return df.groupby("country_id").apply(lambda x: x.ffill().bfill()).reset_index(drop=True)

X = fill_missing(X)
X_test = fill_missing(X_test)

**5. CLASS IMBALANCE**

In [7]:
# =========================
# CLASS WEIGHT
# =========================

pos = y.sum()
neg = len(y) - pos
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 53.573926868044516


**6. MODEL DEFINITIONS**

In [8]:
# =========================
# MODELS
# =========================

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric="Logloss",
    verbose=0,
    random_state=42
)

**7. TRAIN MODELS WITH OOF**

**LIGHTGBM**

In [14]:
# =========================
# LIGHTGBM
# =========================

oof_lgb, test_lgb, scores_lgb, models_lgb = run_time_cv_oof(
    model=lgb_model,
    X=X,
    y=y,
    df=train,
    X_test=X_test,
    model_name="LightGBM"
)


🚀 Running Time-Based CV for: LightGBM

🔹 Fold 1
[LightGBM] [Info] Number of positive: 195, number of negative: 13535
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004587 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2113
[LightGBM] [Info] Number of data points in the train set: 13730, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.014202 -> initscore=-4.240035
[LightGBM] [Info] Start training from score -4.240035
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

**XGBoost**

In [15]:
# =========================
# XGBOOST
# =========================

oof_xgb, test_xgb, scores_xgb, models_xgb = run_time_cv_oof(
    model=xgb_model,
    X=X,
    y=y,
    df=train,
    X_test=X_test,
    model_name="XGBoost"
)


🚀 Running Time-Based CV for: XGBoost

🔹 Fold 1
F1 Score: 0.87857

🔹 Fold 2
F1 Score: 0.90909

🔹 Fold 3
F1 Score: 0.79577

🔹 Fold 4
F1 Score: 0.62662

📊 CV Mean F1 (XGBoost): 0.80252
📉 Std: 0.10969


**CatBoost**

In [16]:
# =========================
# CATBOOST
# =========================

oof_cat, test_cat, scores_cat, models_cat = run_time_cv_oof(
    model=cat_model,
    X=X,
    y=y,
    df=train,
    X_test=X_test,
    model_name="CatBoost"
)


🚀 Running Time-Based CV for: CatBoost

🔹 Fold 1
F1 Score: 0.86466

🔹 Fold 2
F1 Score: 0.92473

🔹 Fold 3
F1 Score: 0.81102

🔹 Fold 4
F1 Score: 0.63809

📊 CV Mean F1 (CatBoost): 0.80963
📉 Std: 0.10689


**8. VALIDATE OOF QUALITY**

In [17]:
# =========================
# OOF VALIDATION
# =========================

def evaluate_oof(name, oof_preds):
    preds = (oof_preds > 0.5).astype(int)
    score = f1_score(y, preds)
    print(f"{name} OOF F1: {score:.5f}")

evaluate_oof("LGB", oof_lgb)
evaluate_oof("XGB", oof_xgb)
evaluate_oof("CAT", oof_cat)

LGB OOF F1: 0.70055
XGB OOF F1: 0.66052
CAT OOF F1: 0.66800


**9. SAVE PREDICTIONS**

In [18]:
# =========================
# SAVE OUTPUTS
# =========================

os.makedirs("D:/AI4EAC- Loan_default_prediction/outputs/oof", exist_ok=True)
os.makedirs("D:/AI4EAC- Loan_default_prediction/outputs/test", exist_ok=True)

np.save("D:/AI4EAC- Loan_default_prediction/outputs/oof/lgb_oof.npy", oof_lgb)
np.save("D:/AI4EAC- Loan_default_prediction/outputs/oof/xgb_oof.npy", oof_xgb)
np.save("D:/AI4EAC- Loan_default_prediction/outputs/oof/cat_oof.npy", oof_cat)

np.save("D:/AI4EAC- Loan_default_prediction/outputs/test/lgb_test.npy", test_lgb)
np.save("D:/AI4EAC- Loan_default_prediction/outputs/test/xgb_test.npy", test_xgb)
np.save("D:/AI4EAC- Loan_default_prediction/outputs/test/cat_test.npy", test_cat)

print("✅ OOF and test predictions saved.")

✅ OOF and test predictions saved.


**10. SAVE FEATURE LIST**

In [19]:
import joblib
joblib.dump(FEATURES, "D:/AI4EAC- Loan_default_prediction/models/final_features_oof_v1.pkl")

['D:/AI4EAC- Loan_default_prediction/models/final_features_oof_v1.pkl']

# 📊 Out-of-Fold Model Training — Professional Summary

## 🎯 Objective

This stage implemented a **production-grade model training pipeline** using time-aware cross-validation and Out-of-Fold (OOF) predictions to ensure realistic performance estimation and enable downstream ensembling.

---

## 🧠 Key Pipeline Decisions

### 1. Feature Consistency & Leakage Prevention

* Removed **train-only features** (`loan_count`, `past_defaults`, etc.) to ensure consistency between train and test.
* Eliminated **non-numeric columns** (e.g., `country`) to maintain model compatibility.
* Enforced **strict feature alignment** using `reindex`, guaranteeing identical feature spaces across datasets.

👉 Result: A **robust, inference-safe feature set (27 features)** suitable for real-world deployment.

---

### 2. Time-Based Cross-Validation

* Implemented **time-aware splits** based on `disbursement_date`.
* Prevented temporal leakage by ensuring validation data always occurs after training data.

👉 Insight:

* This produces a **more realistic estimate of model performance**, especially important given the cross-country (Kenya → Ghana) generalization requirement.

---

### 3. OOF Prediction Framework

* Generated:

  * **OOF predictions** → for unbiased evaluation & stacking
  * **Test predictions** → averaged across folds
* Stored predictions for all models:

  * LightGBM
  * XGBoost
  * CatBoost

👉 This establishes the foundation for:

* Ensemble learning
* Threshold optimization
* Meta-model stacking

---

## 📈 Model Performance Analysis

### 🔹 Cross-Validation (Fold-Based)

| Model    | Mean F1    | Std Dev |
| -------- | ---------- | ------- |
| LightGBM | **0.8225** | 0.0901  |
| XGBoost  | 0.8025     | 0.1097  |
| CatBoost | 0.8096     | 0.1069  |

---

### 🔹 OOF Performance (Global)

| Model    | OOF F1     |
| -------- | ---------- |
| LightGBM | **0.7006** |
| XGBoost  | 0.6605     |
| CatBoost | 0.6680     |

---

## 🧠 Key Insights

### 1. CV vs OOF Gap (CRITICAL)

There is a noticeable drop from CV → OOF:

* LightGBM: **0.82 → 0.70**
* XGBoost: **0.80 → 0.66**
* CatBoost: **0.81 → 0.67**

👉 Interpretation:

* Indicates **strong class imbalance + threshold sensitivity**
* Suggests models are **good at ranking (probabilities)** but default threshold (0.5) is suboptimal

✅ This is expected and will be addressed via:

* Threshold optimization
* Ensemble calibration

---

### 2. Model Stability

* LightGBM shows:

  * Highest mean performance
  * Lowest variance among models

👉 Conclusion:

> **LightGBM is the strongest base learner**

---

### 3. Model Diversity (VERY IMPORTANT FOR ENSEMBLING)

* XGBoost and CatBoost:

  * Slightly lower performance
  * Higher variance
  * Different error patterns

👉 This is GOOD because:

> Ensemble performance depends on **diversity, not just accuracy**

---

### 4. Fold Variability

Observed:

* Some folds perform significantly worse (~0.62–0.64)

👉 Likely causes:

* Temporal distribution shift
* Economic condition variation
* Customer behavior changes over time

👉 Implication:

> Reinforces importance of **time-based CV and macro features**

---

## ⚠️ Observations & Risks

### 🔴 1. Threshold Misalignment

* Using default 0.5 threshold leads to underperformance in OOF

👉 Must fix in next stage

---

### 🔴 2. Class Imbalance

* `scale_pos_weight ≈ 53.57`

👉 Extremely imbalanced dataset:

* Model may under-predict positives
* Precision-recall tradeoff is critical

---

### 🔴 3. LightGBM Warning:

```
No further splits with positive gain
```

👉 Interpretation:

* Some features have limited additional signal
* Model may be saturating early

👉 Not critical, but worth monitoring

---

## ✅ Achievements

✔ Built **fully reproducible OOF pipeline**
✔ Implemented **time-aware validation (competition-grade)**
✔ Ensured **feature consistency & no leakage**
✔ Generated **ensemble-ready predictions**
✔ Identified **performance gaps and optimization opportunities**

---

## 🚀 Next Steps (High Impact Phase)

The next stage will focus on:

### 🔥 1. Threshold Optimization

* Find optimal probability cutoff for F1

### 🔥 2. Model Ensembling

* Weighted averaging
* Rank averaging
* Stacking (meta-model)

### 🔥 3. Model Correlation Analysis

* Identify complementary models

---
